In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####
ω_b = (7)*2pi
ω_a = ω_b / 2

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2 # Impedance of the memory
C_a_inv = Z_a*ω_a # 1/Capacitance of the memory
E_Ca = C_a_inv/2 # Energy of the memory capacitor

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2 # Impedance of the buffer
C_b_inv = Z_b*ω_b # 1/Capacitance of the buffer
E_Cb = C_b_inv/2 # Energy of the buffer capacitor

E_J = 37.12*2*pi


##Drive

ω_d = ω_b
ω_p = 0
ϵ_p = 0.8#5pi
g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env
κ_b = 0.1; # 1/ns from Leon's paper
#κ_2 = 4*g2^2/κ_b
κ_a = κ_b/2;#3e-3*κ_2 #from Leon's paper

# First change of variables
Q_a, Q_b = 1/κ_a, 1/κ_b
Q_a_tilde, Q_b_tilde = ω_a/κ_a, ω_b/κ_b
β_a, β_b = 4E_J*sin(ϵ_p)φ_a^2/(ω_a), 4E_J*sin(ϵ_p)φ_b^2/ω_b
λ_a, λ_b = φ_a, φ_b
ϵ_λ_d = ϵ_d*2*λ_b/ω_b
ν_d = 1;

# Second change of variables
κ_a_tilde, κ_b_tilde = 1/(4*Q_a_tilde*sqrt(1-1/(4Q_b_tilde^2))), 1/(2*Q_b_tilde*sqrt(1-1/(4Q_b_tilde^2)))
β_a_tilde, β_b_tilde = β_a/(2-1/(2Q_b_tilde^2)), β_b/(1-1/(4Q_b_tilde^2))
ϵ_d_tilde = ϵ_λ_d/(1-1/(4Q_b_tilde^2))
ν_d_tilde = ν_d/sqrt(1-1/(4Q_b_tilde^2))
γ = (1-1/(4Q_a_tilde^2))/(1-1/(4Q_b_tilde^2))/2;


In [ ]:
A=[(abs(β_a_tilde)+γ-1/2)/κ_a_tilde abs(β_a_tilde)/sqrt(κ_a_tilde*κ_b_tilde); abs(β_b_tilde)/sqrt(κ_a_tilde*κ_b_tilde) abs(β_b_tilde)/κ_b_tilde]
A_A = A*transpose(A)
eig = sqrt(maximum(eigvals(A_A)))

In [ ]:
sqrt(abs(β_a_tilde^2/κ_a_tilde^2+β_a_tilde^2/(κ_a_tilde*κ_b_tilde)+β_b_tilde^2/κ_b_tilde^2+β_b_tilde^2/(κ_a_tilde*κ_b_tilde)))

In [ ]:
println(2κ_a_tilde," :  ",abs(β_a_tilde+γ-1/2))
println(2κ_b_tilde," :  ",abs(β_b_tilde))
println(sqrt(κ_a_tilde*κ_b_tilde)," :  ",max(β_a_tilde,β_b_tilde))

In [ ]:
sqrt(β_a_tilde^2+(abs(β_b_tilde)+abs(ϵ_d_tilde))^2)/min(κ_b_tilde, κ_a_tilde-abs(γ-1/2)/2)

In [ ]:
σ_p_tilde = 1/2*((γ-1/2+abs(β_a_tilde))^2+β_a_tilde^2+2β_b_tilde^2+sqrt(((γ-1/2+abs(β_a_tilde))^2+β_a_tilde^2+2β_b_tilde^2)^2-4*(γ-1/2)^2*β_b_tilde^2))
n_period = 2
log(exp(2*n_period*pi*sqrt(σ_p_tilde))-2)/(2*n_period*pi)

In [ ]:
t_max = 30/κ_b # Maximal time of the simulation (by default the minimal time is 0.0)
a_0 = 0.0 # Initial value of the memory field
b_0 = 0.0 # Initial value of the buffer field

system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0, b_0) # Initialize the system
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase) # Initialize the drive
environment = MeanField(κ_a, κ_b) # Initialize the environment

a, b, t_list = simulator(t_max, system, drive, environment) # Time series of the memory field, buffer field, list of instants 

a_rot = a.*exp.(1im*ω_a*t_list); # memory field in the rotating frame (where the cat no longer moves)
b_rot = b.*exp.(1im*ω_b*t_list); # memory field in the rotating frame (where the cat no longer moves)

In [ ]:
length(t_list)

In [ ]:
plot(real.(a_rot), imag.(a_rot), label="")
plot!(xlabel=L"\Re(a)", ylabel=L"\Im(a)", size = size_px_car, 
title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",)

In [ ]:
function f(Q, n)
    root = sqrt(1-1/(4*Q^2))
    exp_1 = exp(n*pi*sqrt(2)*sqrt(β_a^2+4β_b^2)/root)
    exp_2 = exp(n*pi*1/Q/root)
    return(exp_1-exp_2)
end
x = 2:0.1:100
y = f.(x, 2);

In [ ]:
exp_1 = exp(2*pi*sqrt(2)*sqrt(β_a^2+4β_b^2))

In [ ]:
plot(x,y)

In [ ]:
8*E_J*sin(0.01)/ω_b*sqrt(φ_a^2+φ_b^2)

In [ ]:
asin(log(2)*ω_b/(sqrt(2)*2*pi*8*E_J))

In [ ]:
asin(1/(4*E_J*sqrt(φ_a^4*Q_a^2+φ_a^4*Q_a*Q_b+φ_b^4*Q_b^2+φ_b^4*Q_a*Q_b)))

In [ ]:
t_max=20/κ_b*ω_b
re_a_0, im_a_0 = -30, -30#-10, -10
x_a_0, p_a_0=sqrt(2)*re_a_0, sqrt(2)*im_a_0
re_b_0, im_b_0 = 30, 30#5, 5
x_b_0, p_b_0=sqrt(2)*re_b_0, sqrt(2)*im_b_0

ϵ_d_p = ϵ_d*sqrt(2)/ω_b
system = SystemATS_Michiel(β_a, β_b, λ_a, λ_b, vcat(x_a_0, x_b_0), vcat(p_a_0, p_b_0))
drive = DriveOneMode(ϵ_d_p, ν_d, buffer_phase)
environment = MeanField(1/Q_a_tilde, 1/Q_b_tilde)

x_a, x_b, p_a, p_b, t_list = simulator(t_max, system, drive, environment; equation=default_equation!)
a = @. (x_a+1im*p_a)/sqrt(2)
a_rot = a.*exp.(1im*t_list/2)

b = @. (x_b+1im*p_b)/sqrt(2)
b_rot = b.*exp.(1im*t_list);

In [ ]:
x_lam_a, x_lam_b, p_lam_a, p_lam_b = sqrt(2)*λ_a*x_a, sqrt(2)*λ_b*x_b, sqrt(2)*λ_a*p_a, sqrt(2)*λ_b*p_b
x_a_tilde, x_b_tilde, p_a_tilde, p_b_tilde = x_lam_a, x_lam_b, (p_lam_a+x_lam_a/(2*Q_a_tilde))/sqrt(1-1/(4*Q_b_tilde^2)), (p_lam_b+x_lam_b/(2*Q_b_tilde))/sqrt(1-1/(4*Q_b_tilde^2))
s = sqrt(1-1/(4*Q_b_tilde^2))*t_list
u_a =@. cos(t_list/2)*x_a_tilde - sin(t_list/2)*p_a_tilde
v_a =@. sin(t_list/2)*x_a_tilde + cos(t_list/2)*p_a_tilde
u_b =@. cos(t_list)*x_b_tilde - sin(t_list)*p_b_tilde
v_b =@. sin(t_list)*x_b_tilde + cos(t_list)*p_b_tilde;

a_R = (u_a+1im*v_a)/sqrt(2)
b_R = (u_b+1im*v_b)/sqrt(2);

In [ ]:
b_focus = b_R[120000:100:end]
plot(real.(b_focus),imag.(b_focus))

In [ ]:
a_focus = a_R[120000:100:end]
plot(real.(a_focus),imag.(a_focus))

In [ ]:
#a_R_target = 1im*(0.675-0.17*1im)
#b_R_target = 1im*(0.5-0.25*1im)
a_R_target = 0.006-0.255
b_R_target = (-0.045-0.02*1im)

rhs_1 = β_a_tilde*besselj(1,sqrt(2)*abs(b_R_target))*exp(-1im*angle(a_R_target/b_R_target))*(besselj(1,sqrt(2)*abs(a_R_target))-besselj(3,sqrt(2)*abs(a_R_target))*exp(2*1im*angle(a_R_target^2/b_R_target)))
lhs_1 = -1im*sqrt(2)*κ_a_tilde*a_R_target

rhs_2 = β_b_tilde*besselj(2,sqrt(2)*abs(a_R_target))*exp(-2*1im*angle(a_R_target/b_R_target))*(besselj(0,sqrt(2)*abs(b_R_target))*exp(2*1im*angle(a_R_target^2/b_R_target))-besselj(2,sqrt(2)*abs(b_R_target)))
lhs_2 = ϵ_d_tilde/2 - 1im*sqrt(2)*κ_b_tilde*b_R_target;

In [ ]:
lhs_1

In [ ]:
rhs_1

In [ ]:
R =@. sqrt(u_a^2+v_a^2+u_b^2+v_b^2);

In [ ]:
plot(t_list, R)

In [ ]:
besselj(1,0.8)

In [ ]:
dephasing = -4*E_J*sin(ϵ_p)*(φ_a^2 + φ_b^2/2)/ω_a
δ = 1-ν_d_tilde/2

M =[-κ_a_tilde δ+β_a_tilde*sin(dephasing)/2 0 β_a_tilde*sin(dephasing)/2; -(δ+β_a_tilde*sin(dephasing)/2) -κ_a_tilde -β_a_tilde*sin(dephasing)/2 0; 0 β_b_tilde*sin(dephasing)/2 -κ_b_tilde δ+β_b_tilde*sin(dephasing)/2; -β_b_tilde*sin(dephasing)/2 0 -(δ+β_b_tilde*sin(dephasing)/2) -κ_b_tilde]

In [ ]:
N = nullspace(M)

# Bessel play

In [ ]:
x_bess = 0:0.01:1
y_bess = besselj.(0,x_bess)
plot(x_bess,y_bess)
plot!(x_bess, 1 .-x_bess .^2 ./4)


In [ ]:
C = 2*sqrt(2)*κ_a_tilde/(1im*β_a_tilde)
D = 1im*β_b_tilde*besselj(0,sqrt(2)*abs(C))/sqrt(2)

r_test = 0:0.001:5
y_test =@. abs(-κ_b_tilde*C+D*besselj(2,sqrt(2)*r_test)) - abs(ϵ_d_tilde)/2/sqrt(2)

plot(r_test,y_test)

In [ ]:
plot(0:0.001:6,besselj.(1,(0:0.001:6).*sqrt(2))-besselj.(3,(0:0.001:6).*sqrt(2)))
vline!([2.7])
vline!([2.15])

In [ ]:
sqrt(abs(2*(ϵ_d_tilde-2im*sqrt(2)*κ_b_tilde*2.709)/(β_b_tilde*besselj(0,sqrt(2)*2.709))))

In [ ]:
sqrt(abs(2*(ϵ_d_tilde-2im*sqrt(2)*κ_b_tilde*0)/(β_b_tilde*besselj(0,sqrt(2)*0))))

In [ ]:
r_test = 0:0.001:5
y_test =@. -κ_b_tilde*2.7+β_b_tilde*besselj(2,sqrt(2)*r_test)*besselj(0,sqrt(2)*2.7)/sqrt(2) - ϵ_d_tilde/2/sqrt(2)
plot(r_test,y_test)

In [ ]:
sqrt(4*sqrt(2)/(β_b_tilde*besselj(0,sqrt(2)*0.56))*(ϵ_d_tilde/(2*sqrt(2))-1im*κ_b_tilde*0.56))

In [ ]:
sqrt(4*sqrt(2)/(β_b_tilde*besselj(0,4sqrt(2)*0.56))*(ϵ_d_tilde/(2*sqrt(2))-1im*κ_b_tilde*0.56))

In [ ]:
sqrt(4*sqrt(2)/(β_b_tilde*besselj(0,4sqrt(2)*0.56))*(ϵ_d_tilde/(2*sqrt(2))-1im*κ_b_tilde*0.56*4))

# Root finding

In [ ]:
n_max = 2
function expr_1(a::ComplexF64,b::ComplexF64)
    b=b*pi
    return -κ_a_tilde*a+1im*β_a_tilde/sqrt(2)*sum((-1)^k*besselj(4k+3,sqrt(2)*abs(a))*besselj(2k+1,sqrt(2)*abs(b))*exp(1im*((4k+3)*angle(a)-(2k+1)*angle(b))) for k in -n_max:1:n_max)
end

function expr_2(a::ComplexF64,b::ComplexF64)
    b=b*pi
    return -κ_b_tilde*b+1im*β_b_tilde/sqrt(2)*sum((-1)^(k+1)*besselj(4k+2,sqrt(2)*abs(a))*besselj(2k,sqrt(2)*abs(b))*exp(1im*((4k+2)*angle(a)-(2k)*angle(b))) for k in -n_max:1:n_max)-1im*ϵ_d_tilde/2/sqrt(2)
end

In [ ]:
re_a_list = -1:0.1:-0.5
im_a_list = 0:0.1:0.3
re_b_list = 0.2:0.1:0.8
im_b_list = -0.6:0.1:0

a_grid = re_a_list .+ 1im .* im_a_list'           # size (Na, Ni_a)
b_grid = re_b_list .+ 1im .* im_b_list'           # size (Nb, Ni_b)

expr_1_list = Array{Float64}(undef,
    length(re_a_list), length(im_a_list),
    length(re_b_list), length(im_b_list)
)
expr_2_list = Array{Float64}(undef,
    length(re_a_list), length(im_a_list),
    length(re_b_list), length(im_b_list)
)

@inbounds for ia in axes(a_grid, 1), ja in axes(a_grid, 2)
    a = a_grid[ia, ja]
    for ib in axes(b_grid, 1), jb in axes(b_grid, 2)
        expr_1_list[ia, ja, ib, jb] = abs(expr_1(a, b_grid[ib, jb]))
        expr_2_list[ia, ja, ib, jb] = abs(expr_2(a, b_grid[ib, jb]))
    end
end


In [ ]:
argmin(expr_2_list)